## **IROS Coding Noise Reduction Efficiency**

In [6]:
from pathlib import Path
from typing import Any, Optional

import numpy as np
from numpy.typing import NDArray
import matplotlib as mpl
import matplotlib.pyplot as plt

import darksun as ds
import imgmaker as mgm

plt.rcdefaults()

RCPARAMS = {
    'axes.formatter.limits': (-3, 3),
    'savefig.bbox': 'tight',
}
mpl.rcParams.update(RCPARAMS)

In [ ]:
BASEDIR: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Outputs"

SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks_mask25"
RUN_ID: str = "GC_smoothed_rec_1ks_2-50keV_mask25"
ID_CAMERA: str = "cam1a"

DIRPATH: str = f"{BASEDIR}/Out{SKYFIELD}/{DATA_FITS}/GC_rec_1ks_2-50keV_mask25"
DIRPATH_SMOOTHED: str = f"{BASEDIR}/Out{SKYFIELD}/{DATA_FITS}/GC_smoothed_rec_1ks_2-50keV_mask25"

OUT_RESULTS_PATH = mgm.config_savedata_to()

# base IROS run
_, base_start_snr = ds.load_sky(f'{DIRPATH}/sky_SIMUL_{ID_CAMERA.upper()}.fits')
_, base_res_snr = ds.load_sky(f'{DIRPATH}/skyRES_IROS_{ID_CAMERA.upper()}.fits')
# BKG subtracted IROS run
_, smthd_start_snr = ds.load_sky(f'{DIRPATH_SMOOTHED}/sky_SIMUL_{ID_CAMERA.upper()}.fits')
_, smthd_res_snr = ds.load_sky(f'{DIRPATH_SMOOTHED}/skyRES_IROS_{ID_CAMERA.upper()}.fits')

In [15]:
def get_hists(
    start: NDArray,
    res: NDArray,
    nbins: int | NDArray = 1000,
    density: bool = True,
    distr_lims: Optional[tuple[float, float]] = (None, None),
    img_cut: Optional[tuple[int, int]] = (None, None),
) -> tuple[NDArray, NDArray, NDArray]:
    """
    Handles sky images statistics processing, returning IROS 1st/last iter. distributions.
    """
    start, res = map(lambda x: ds.unframe(x, *img_cut), (start, res))

    hist_start, bins = np.histogram(start, bins=nbins, density=density)
    hist_res, _ = np.histogram(res, bins=bins, density=density)
    bins_slice, hist_slice = ds.config_distr_limits(bins, *distr_lims)

    return bins[bins_slice], hist_start[hist_slice], hist_res[hist_slice]

def show_snrDistr_evol(
    hist_start: NDArray,
    hist_last_base: NDArray,
    hist_last_smthd: NDArray,
    bins: NDArray,
    save_to: str | Path | None = None,
    save_options: dict[str, Any] | None = None,
    **fig_kws,
) -> None:
    """Plots the SNR distr. evolution at 1st and last IROS iters."""
    nb = len(bins)
    x = bins[nb // 3 : -nb // 3 - 1]
    stdgauss = 1.0 / np.sqrt(2 * np.pi) * np.exp(-np.square(x) / 2)

    fig, ax = plt.subplots(1, 1, tight_layout=True, **fig_kws)
    ax.plot(x, stdgauss, "#0D0D0D", linestyle='--', alpha=0.2)
    ax.stairs(hist_start, edges=bins, edgecolor='OrangeRed', label='1$^{st}$ iter.')
    ax.stairs(hist_last_base, edges=bins, edgecolor='DodgerBlue', label='last iter.')
    # ax.stairs(hist_1st_smthd, edges=bins, edgecolor='g', label='1st it. (bkg-sub)', alpha=0.85)
    ax.stairs(hist_last_smthd, edges=bins, edgecolor='Violet', label='last (wto bkg)', alpha=0.9)
    ax.set_title('Coding Noise Reduction Efficiency', pad=10)
    ax.set_xlabel(xlabel='SNR')
    ax.set_ylabel(ylabel='density')
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')
    ax.tick_params(which='both', direction='in', width=1.5, length=4 if 'major' else 2.5)
    # ax.set_yscale('log')
    ax.legend(loc='best')
    if save_to is not None:
        save_options_ = save_options or {}
        fig.savefig(save_to, **save_options_)
    plt.show()
    return

In [48]:
# upsampling factors for underlying binning grids
UPS_X, UPS_Y = 2, 1

# SNR distr. vs coding noise reduction (true sky --> last IROS iteration)
num_bins = 1000
distr_lims = (-40, 40)
img_cut = (UPS_Y * 30, UPS_X * 20)

# base IROS run
bins, hist_base_start_snr, hist_base_res_snr = get_hists(
    base_start_snr, base_res_snr, nbins=num_bins, distr_lims=distr_lims, img_cut=img_cut,
)

# BKG subtracted IROS run
_, _, hist_smthd_res_snr = get_hists(
    smthd_start_snr, smthd_res_snr, nbins=bins, distr_lims=distr_lims, img_cut=img_cut,
)

In [ ]:
show_snrDistr_evol(
    hist_start=hist_base_start_snr,
    hist_last_base=hist_base_res_snr,
    hist_last_smthd=hist_smthd_res_snr,
    bins=bins,
    # save_to=f'{OUT_RESULTS_PATH}/snr_distr_vs_codnoise_wbkg_sub.svg',
    figsize=(3.5, 3.5),
)

In [56]:
from scipy.optimize import curve_fit

def Gaussian(x: NDArray, x0: float, sigma: float) -> NDArray:
    """Returns a 1D Gaussian distribution centered around `x0` and with std `sigma`."""
    return (
        1.0 / np.sqrt(2 * np.pi * np.square(sigma)) * np.exp(-0.5 * np.square((x - x0) / sigma))
    )

def print_fitres(case: str, popt: NDArray, pcov: NDArray) -> NDArray:
    mu, sigma = map(float, popt)
    dmu, dsigma = map(float, np.sqrt(np.diag(pcov)))
    print(f"{case}: {mu:.5f} +/- {dmu:.5f}  |  {sigma:.5f} +/- {dsigma:.5f}")
    return

# fit of the residual SNR distr.
center_bins = 0.5 * (bins[:-1] + bins[1:])

popt_base, pcov_base = curve_fit(Gaussian, xdata=center_bins, ydata=hist_base_res_snr, p0=[0, 1.0])
popt_smthd, pcov_smthd = curve_fit(Gaussian, xdata=center_bins, ydata=hist_smthd_res_snr, p0=[0, 1.0])

print_fitres('Original (mu, sigma)', popt_base, pcov_base)
print_fitres('Smoothed (mu, sigma)', popt_smthd, pcov_smthd)

Original (mu, sigma): 0.00302 +/- 0.00074  |  1.11854 +/- 0.00061
Smoothed (mu, sigma): -0.00129 +/- 0.00043  |  1.06838 +/- 0.00035
